In [ ]:
import csv
import sys
from pathlib import Path

import numpy as np
import torch

script_dir = Path.cwd()
if not (script_dir / "utils" / "sls_controlled_experiment.py").exists():
    candidate = Path("ad_detection/train_notebook/SLS").resolve()
    if candidate.exists():
        script_dir = candidate
sys.path.insert(0, str(script_dir.parent.parent / "train"))
sys.path.insert(0, str(script_dir / "utils"))

from SLS_Model.model import AD_SLS_Model
from SLS_Model.train import train, validate
from utils.dataset import create_dataloaders
from utils.visualization import plot_training_curves
from XLSR_model.model import SSLModel

from sls_controlled_experiment import (
    bootstrap_binary_metrics_with_samples,
    compute_binary_metrics,
    create_profile_repeat_split,
    ensure_sls_features_for_split,
    evaluate_lu_with_predictions,
    flatten_bootstrap_summary,
    save_dict_rows,
    summarize_metric_rows,
    validate_profile_availability,
)

SLS_BATCH_SIZE = 4


def load_dict_rows(path: Path) -> list[dict[str, str]]:
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        return list(csv.DictReader(file))


In [ ]:
BASE_DATASET_NAME = "Pitt-origin"
TARGET_PROFILE = "ADReSS-like"
N_REPEATS = 10
REPEAT_SEEDS = [2026 + i for i in range(N_REPEATS)]
TRAIN_SEEDS = [21, 42, 84, 168, 336]
BOOTSTRAP_REPEATS = 1000
LU_SAMPLE_SIZE = 74
TRAIN_RATIO = 0.8

MODEL_OUTPUT_DIR = script_dir.parent.parent / "models" / f"{TARGET_PROFILE}_sls_controlled_experiment"
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using {device}")


## Step 1: Validate Target Profile Availability


In [ ]:
profile_availability = validate_profile_availability(TARGET_PROFILE)
print(profile_availability)


## Step 2: Train Matched Pitt-origin Subsets

This cell only trains models and saves training artifacts. It does not test Lu.


In [ ]:
TRAINING_FIELDS = [
    "dataset_name",
    "seed",
    "acc",
    "control_acc",
    "dementia_acc",
    "macro_f1",
    "control_f1",
    "dementia_f1",
]
LU_RESULTS_DIR = MODEL_OUTPUT_DIR / "lu_test_results"
LU_RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def save_training_results(path: Path, rows: list[dict[str, object]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=TRAINING_FIELDS, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def evaluate_checkpoint_metrics(checkpoint_path: Path, val_loader, device) -> dict[str, float]:
    import torch.nn.functional as F

    model = AD_SLS_Model().to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint)
    model.eval()

    total_loss = 0
    y_true = []
    y_pred = []
    with torch.no_grad():
        for features, labels, masks in val_loader:
            features = features.to(device)
            labels = labels.to(device)
            masks = masks.to(device)
            logits = model(features, masks)
            total_loss += F.cross_entropy(logits, labels).item()
            predictions = torch.argmax(logits, dim=1)
            y_true.extend(labels.detach().cpu().tolist())
            y_pred.extend(predictions.detach().cpu().tolist())

    metrics = compute_binary_metrics(y_true, y_pred)
    del model
    return {
        "val_acc": metrics["accuracy"],
        "val_loss": total_loss / len(val_loader),
        "control_acc": metrics["control_acc"],
        "dementia_acc": metrics["dementia_acc"],
        "macro_f1": (metrics["control_f1"] + metrics["dementia_f1"]) / 2,
        "control_f1": metrics["control_f1"],
        "dementia_f1": metrics["dementia_f1"],
        "f1_score": metrics["dementia_f1"],
    }


split_summaries = []
training_rows = []
ssl_model = None

for repeat_idx, repeat_seed in enumerate(REPEAT_SEEDS):
    print("\n" + "=" * 90)
    print(f"Training repeat {repeat_idx + 1}/{N_REPEATS} | profile={TARGET_PROFILE} | subset_seed={repeat_seed}")
    print("=" * 90)

    train_csv, val_csv, split_summary = create_profile_repeat_split(
        profile_name=TARGET_PROFILE,
        repeat_idx=repeat_idx,
        repeat_seed=repeat_seed,
        train_ratio=TRAIN_RATIO,
    )
    sampled_rows = split_summary.pop("sampled_rows")
    split_summaries.append(split_summary)

    repeat_output_dir = MODEL_OUTPUT_DIR / f"repeat_{repeat_idx:02d}"
    save_dict_rows(repeat_output_dir / "subset_samples.csv", sampled_rows)

    print(f"Train CSV: {train_csv}")
    print(f"Val CSV: {val_csv}")
    print(f"Split summary: total={split_summary['total']}, train={split_summary['train_size']}, val={split_summary['val_size']}")

    ssl_model = ensure_sls_features_for_split(
        train_csv=train_csv,
        val_csv=val_csv,
        device=device,
        ssl_model=ssl_model,
    )

    train_loader = create_dataloaders(
        data_csv=train_csv,
        feature_type="sls",
        batch_size=SLS_BATCH_SIZE,
    )
    val_loader = create_dataloaders(
        data_csv=val_csv,
        feature_type="sls",
        batch_size=SLS_BATCH_SIZE,
    )

    seed_results_path = repeat_output_dir / "seed_results.csv"
    seed_result_by_seed = {}

    for train_seed in TRAIN_SEEDS:
        best_model_path = repeat_output_dir / f"seed_{train_seed}" / "best.pth"
        history_path = repeat_output_dir / f"training_history_seed_{train_seed}.csv"

        if best_model_path.exists():
            print(f"Skip training seed {train_seed}: found existing best model at {best_model_path}")
            history = None
        else:
            seed, _, history = train(
                seed=train_seed,
                train_loader=train_loader,
                val_loader=val_loader,
                output_dir=repeat_output_dir,
                device=device,
            )
            train_seed = seed
            best_model_path = repeat_output_dir / f"seed_{train_seed}" / "best.pth"

        metrics = evaluate_checkpoint_metrics(best_model_path, val_loader, device)
        seed_result_by_seed[train_seed] = {
            "repeat_idx": repeat_idx,
            "subset_seed": repeat_seed,
            "seed": train_seed,
            "val_acc": metrics["val_acc"],
            "val_loss": metrics["val_loss"],
            "control_acc": metrics["control_acc"],
            "dementia_acc": metrics["dementia_acc"],
            "macro_f1": metrics["macro_f1"],
            "control_f1": metrics["control_f1"],
            "dementia_f1": metrics["dementia_f1"],
            "f1_score": metrics["f1_score"],
        }
        training_rows.append({
            "dataset_name": f"{TARGET_PROFILE}-repeat_{repeat_idx:02d}",
            "seed": train_seed,
            "acc": f"{metrics['val_acc']:.4f}",
            "control_acc": f"{metrics['control_acc']:.4f}",
            "dementia_acc": f"{metrics['dementia_acc']:.4f}",
            "macro_f1": f"{metrics['macro_f1']:.4f}",
            "control_f1": f"{metrics['control_f1']:.4f}",
            "dementia_f1": f"{metrics['dementia_f1']:.4f}",
        })

        if history is not None:
            history_rows = [
                {
                    "repeat_idx": repeat_idx,
                    "subset_seed": repeat_seed,
                    "seed": train_seed,
                    "epoch": epoch,
                    "train_loss": train_loss,
                    "val_loss": val_loss,
                    "train_acc": train_acc,
                    "val_acc": val_acc,
                }
                for epoch, train_loss, val_loss, train_acc, val_acc in zip(
                    history["epochs"],
                    history["train_losses"],
                    history["val_losses"],
                    history["train_accs"],
                    history["val_accs"],
                )
            ]
            save_dict_rows(history_path, history_rows)

            plot_training_curves(
                epochs=history["epochs"],
                train_loss=history["train_losses"],
                val_loss=history["val_losses"],
                train_acc=history["train_accs"],
                val_acc=history["val_accs"],
                title_prefix=f"Repeat {repeat_idx} Seed {train_seed}",
            )

    seed_result_rows = [seed_result_by_seed[seed] for seed in TRAIN_SEEDS if seed in seed_result_by_seed]
    save_dict_rows(seed_results_path, seed_result_rows)
    print(f"Saved seed results to {seed_results_path}")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

save_dict_rows(MODEL_OUTPUT_DIR / "split_summaries.csv", split_summaries)
save_training_results(LU_RESULTS_DIR / "training_results.csv", training_rows)
print(f"Saved split summaries to {MODEL_OUTPUT_DIR / 'split_summaries.csv'}")
print(f"Saved training results to {LU_RESULTS_DIR / 'training_results.csv'}")


## Step 3: Test Saved Models on Lu

This cell is independent of Step 2 memory state. After training finishes, it can be run after a kernel restart as long as the import/config/device cells above have been run.


In [ ]:
LU_TEST_DATASETS = [
    ("Lu", script_dir.parent.parent / "data/raw/Lu"),
    ("Lu-Denoiser", script_dir.parent.parent / "data/denoised/Lu-Denoiser"),
    ("Lu-FRCRN_SE", script_dir.parent.parent / "data/denoised/Lu-FRCRN_SE"),
    ("Lu-MossFormer", script_dir.parent.parent / "data/denoised/Lu-MossFormer"),
    ("Lu-Resemble", script_dir.parent.parent / "data/denoised/Lu-Resemble"),
    ("Lu-MAP-SEMamba", script_dir.parent.parent / "data/denoised/Lu-MAP-SEMamba"),
]

OUTPUT_FIELDS = ["dataset", "acc", "control_acc", "dementia_acc", "macro_f1", "control_f1", "dementia_f1"]
METRIC_KEYS = OUTPUT_FIELDS[1:]
LU_RESULTS_DIR = MODEL_OUTPUT_DIR / "lu_test_results"
LU_RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def dataset_result_dir(dataset_name: str) -> Path:
    return LU_RESULTS_DIR / dataset_name


def bootstrap_path(dataset_name: str, repeat_idx: int) -> Path:
    output_dir = dataset_result_dir(dataset_name) / f"repeat_{repeat_idx:02d}"
    output_dir.mkdir(parents=True, exist_ok=True)
    return output_dir / f"{dataset_name}_bootstrap_samples.csv"


def metric_value(row: dict[str, object], key: str) -> float:
    if key == "acc":
        return float(row["accuracy"])
    if key == "macro_f1":
        return (float(row["control_f1"]) + float(row["dementia_f1"])) / 2
    return float(row[key])


def format_metric_row(dataset_name: str, row: dict[str, object]) -> dict[str, str]:
    return {
        "dataset": dataset_name,
        "acc": f"{metric_value(row, 'acc'):.4f}",
        "control_acc": f"{metric_value(row, 'control_acc'):.4f}",
        "dementia_acc": f"{metric_value(row, 'dementia_acc'):.4f}",
        "macro_f1": f"{metric_value(row, 'macro_f1'):.4f}",
        "control_f1": f"{metric_value(row, 'control_f1'):.4f}",
        "dementia_f1": f"{metric_value(row, 'dementia_f1'):.4f}",
    }


def save_metric_rows(path: Path, rows: list[dict[str, object]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=OUTPUT_FIELDS, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def bootstrap_summary_from_rows(rows: list[dict[str, str]]) -> dict[str, dict[str, float]]:
    summary = {}
    for key in METRIC_KEYS:
        values = np.asarray([float(row[key]) for row in rows], dtype=float)
        values = values[~np.isnan(values)]
        summary[key] = {
            "mean": float(np.mean(values)),
            "std": float(np.std(values, ddof=1)),
            "ci95_low": float(np.percentile(values, 2.5)),
            "ci95_high": float(np.percentile(values, 97.5)),
        }
    return summary


repeat_results = []
bootstrap_results = []
ssl_model = SSLModel(device, freeze_xlsr=True)

for repeat_idx, repeat_seed in enumerate(REPEAT_SEEDS):
    print("\n" + "=" * 90)
    print(f"Testing repeat {repeat_idx + 1}/{N_REPEATS} | profile={TARGET_PROFILE} | subset_seed={repeat_seed}")
    print("=" * 90)

    repeat_output_dir = MODEL_OUTPUT_DIR / f"repeat_{repeat_idx:02d}"
    seed_results_path = repeat_output_dir / "seed_results.csv"
    if not seed_results_path.exists():
        raise FileNotFoundError(f"Missing training results: {seed_results_path}")

    seed_rows = load_dict_rows(seed_results_path)
    best_seed_row = max(
        seed_rows,
        key=lambda row: (float(row["val_acc"]), -float(row["val_loss"])),
    )
    best_seed = int(best_seed_row["seed"])
    best_model_path = repeat_output_dir / f"seed_{best_seed}" / "best.pth"
    if not best_model_path.exists():
        raise FileNotFoundError(f"Missing best model checkpoint: {best_model_path}")
    print(f"Best seed: {best_seed}, best model: {best_model_path}")

    test_model = AD_SLS_Model()
    checkpoint = torch.load(best_model_path, map_location=device)
    test_model.load_state_dict(checkpoint)
    test_model = test_model.to(device)
    test_model.eval()

    for dataset_name, audio_dir in LU_TEST_DATASETS:
        dataset_label = f"{dataset_name}-repeat_{repeat_idx:02d}"
        bootstrap_csv = bootstrap_path(dataset_name, repeat_idx)
        lu_result = evaluate_lu_with_predictions(
            model=test_model,
            device=device,
            ssl_model=ssl_model,
            batch_size=SLS_BATCH_SIZE,
            dataset_name=dataset_name,
            audio_dir=audio_dir,
            feature_dir_name=f"{dataset_name}_sls_features",
        )
        if lu_result["n_samples"] != LU_SAMPLE_SIZE:
            print(
                f"Warning: {dataset_name} prediction count is {lu_result['n_samples']}; "
                f"bootstrap sample size is configured as {LU_SAMPLE_SIZE}."
            )

        _, raw_bootstrap_sample_rows = bootstrap_binary_metrics_with_samples(
            y_true=lu_result["y_true"],
            y_pred=lu_result["y_pred"],
            n_bootstrap=BOOTSTRAP_REPEATS,
            sample_size=LU_SAMPLE_SIZE,
            seed=repeat_seed,
        )
        bootstrap_sample_rows = [
            format_metric_row(dataset_label, row)
            for row in raw_bootstrap_sample_rows
        ]
        save_metric_rows(bootstrap_csv, bootstrap_sample_rows)
        bootstrap_summary = bootstrap_summary_from_rows(bootstrap_sample_rows)

        repeat_row = format_metric_row(dataset_label, lu_result)
        repeat_results.append(repeat_row)

        bootstrap_row = {"dataset": dataset_label}
        for key in METRIC_KEYS:
            mean = bootstrap_summary[key]["mean"]
            std = bootstrap_summary[key]["std"]
            bootstrap_row[key] = f"{mean:.4f}±{std:.4f}"
        bootstrap_results.append(bootstrap_row)

        print(f"{dataset_label} result: {repeat_row}")
        print(f"{dataset_label} bootstrap summary: {bootstrap_summary}")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

save_metric_rows(LU_RESULTS_DIR / "test_results.csv", repeat_results)
save_metric_rows(LU_RESULTS_DIR / "bootstrap_results.csv", bootstrap_results)
print(f"Saved test results to {LU_RESULTS_DIR / 'test_results.csv'}")
print(f"Saved bootstrap results to {LU_RESULTS_DIR / 'bootstrap_results.csv'}")


## Step 4: Summarize Saved Test Results


In [ ]:
test_results_path = LU_RESULTS_DIR / "test_results.csv"
if not test_results_path.exists():
    raise FileNotFoundError(f"Missing test results: {test_results_path}")

test_results = load_dict_rows(test_results_path)

print("\n" + "=" * 90)
print("TEST RESULTS")
print("=" * 90)
for row in test_results:
    print(f"\n[{row['dataset']}]")
    for key in OUTPUT_FIELDS[1:]:
        print(f"{key}: {float(row[key]):.4f}")

test_results
